In [ ]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
# Index
indices = ["^HSI", "^GSPC", "^IXIC", "QLD", "TQQQ"]
index_dict = {"^HSI": "HSI", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite", "QLD": "ProShares Ultra QQQ", "TQQQ": "ProShares UltraPro QQQ"}

# Momentum ETFs
etfs = ["MTUM", "IMTM", "SMLF", "SPMO", "XMMO", 
        "XSMO", "PDP", "IDMO", "JMOM", "GLOV", 
        "VFMO", "DWMF", "AFLG", "3070.HK"]
etfs_dict = {
    "MTUM": "iShares MSCI USA Momentum Factor ETF",
    "IMTM": "iShares MSCI Intl Momentum Factor ETF",
    "SMLF": "iShares U.S. Small‑Cap Equity Factor ETF",
    "SPMO": "Invesco S&P 500 Momentum ETF",
    "XMMO": "Invesco S&P MidCap Momentum ETF",
    "XSMO": "Invesco S&P SmallCap Momentum ETF",
    "PDP": "Invesco Dorsey Wright Momentum ETF",
    "IDMO": "Invesco S&P Intl Developed Momentum ETF",
    "JMOM": "JPMorgan U.S. Momentum Factor ETF",
    "GLOV": "Activebeta World Low Vol Plus Equity ETF",
    "VFMO": "Vanguard U.S. Momentum Factor ETF",
    "DWMF": "WisdomTree International Multifactor ETF",
    "AFLG": "First Trust Active Factor Large Cap Intl ETF",
    "3070.HK": "Ping An of China CSI HK Dividend ETF"
}

In [ ]:
# Create an empty DataFrame to store the data of the ETFs
etfs_data = pd.DataFrame(columns=[
    "Symbol", "Establishment Date", "1 Year CAGR (%)", "3 Year CAGR (%)", 
    "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)", "Max Period CAGR (%)", 
    "Max Period Annual Volatility (%)", "Max Period Sharpe", 
    "Max Period Sortino", "Max Period MDD (%)", "Max Period Calmar", 
    "5 Year Annual Volatility (%)", "5 Year Sharpe", "5 Year Sortino", 
    "5 Year MDD (%)", "5 Year Calmar", "5 Year Peak Date", 
    "5 Year Recovery Date", "5 Year Recovery Period (days)"
])

# Loop through each ETF in the indices and momentum ETFs
for etf in tqdm(indices + etfs):
    # Get the name of the ETF
    etf_name = {**index_dict, **etfs_dict}.get(etf, etf)
    etfs_data.loc[etf_name, "Symbol"] = etf

    # Get the dataframe for the ETF
    df = get_df(etf, current_date, max_period=True, adj=True)
    
    # Get the establishment date of the ETF
    establishment_date = df.index[0].date()
    etfs_data.loc[etf_name, "Establishment Date"] = establishment_date
    
    # Calculate CAGR values for different periods
    for period, label in zip([252, 252 * 3, 252 * 5, 252 * 10, 252 * 20], 
                             ["1 Year CAGR (%)", "3 Year CAGR (%)", "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)"]):
        if len(df) >= period:
            start_price = df["Close"].iloc[- period]
            end_price = df["Close"].iloc[- 1]
            cagr = ((end_price / start_price) ** (1 / (period / 252)) - 1) * 100
            etfs_data.loc[etf_name, label] = cagr

    # Calculate CAGR of maximum period
    start_price = df["Close"].iloc[0]
    end_price = df["Close"].iloc[- 1]
    max_period_cagr = ((end_price / start_price) ** (1 / (len(df) / 252)) - 1) * 100
    etfs_data.loc[etf_name, "Max Period CAGR (%)"] = max_period_cagr

    # Calculate percent change and cumulative return
    df["Percent Change"] = df["Close"].pct_change().dropna()
    df["Cumulative Return"] = (1 + df["Percent Change"]).cumprod()
    daily_returns = df["Percent Change"]
    cumulative_returns = df["Cumulative Return"]
        
    # Calculate annual volatility
    annual_volatility = daily_returns.std() * np.sqrt(252) * 100
    etfs_data.loc[etf_name, "Max Period Annual Volatility (%)"] = annual_volatility

    # Calculate 5 year annual volatility
    if len(df) >= 252 * 5:
        daily_returns_5y = daily_returns.iloc[- 252 * 5:]
        annual_volatility_5y = daily_returns_5y.std() * np.sqrt(252) * 100
        etfs_data.loc[etf_name, "5 Year Annual Volatility (%)"] = annual_volatility_5y

    # Calculate Sharpe ratio
    risk_free_rate = 0.03  # Assuming a risk-free rate of 3%
    sharpe_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (daily_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sharpe"] = sharpe_ratio

    # Calculate 5 year Sharpe ratio
    if len(df) >= 252 * 5:
        sharpe_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (daily_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sharpe"] = sharpe_ratio_5y

    # Calculate Sortino ratio
    negative_returns = daily_returns[daily_returns < 0]
    sortino_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (negative_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sortino"] = sortino_ratio

    # Calculate 5 year Sortino ratio
    if len(df) >= 252 * 5:
        negative_returns_5y = daily_returns_5y[daily_returns_5y < 0]
        sortino_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (negative_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sortino"] = sortino_ratio_5y

    # Calculate maximum drawdown
    df["Peak"] = df["Cumulative Return"].cummax()
    df["Drawdown"] = (df["Cumulative Return"] - df["Peak"]) / df["Peak"]
    drawdowns = df["Drawdown"]
    mdd = drawdowns.min() * 100
    mdd_date = drawdowns.idxmin()
    etfs_data.loc[etf_name, "Max Period MDD (%)"] = mdd

    # Calculate 5 year maximum drawdown
    if len(df) >= 252 * 5:
        drawdowns_5y = drawdowns[- 252 * 5:]
        mdd_5y = drawdowns_5y.min() * 100
        etfs_data.loc[etf_name, "5 Year MDD (%)"] = mdd_5y
        mdd_date_5y = drawdowns_5y.idxmin()

        # Find the peak date before the maximum drawdown
        peak_mask_5y = (df.index <= mdd_date_5y) & (df["Cumulative Return"] == df["Peak"])
        peak_date_5y = df.index[peak_mask_5y][-1]
        etfs_data.loc[etf_name, "5 Year Peak Date"] = peak_date_5y.date()

        # Find the recovery date after the maximum drawdown
        rec_mask_5y = (df.index > mdd_date_5y) & (df["Cumulative Return"] >= df.loc[peak_date_5y, "Cumulative Return"])
        rec_date_5y = df.index[rec_mask_5y][0] if len(df.index[rec_mask_5y]) > 0 else None
        if rec_date_5y is not None:
            etfs_data.loc[etf_name, "5 Year Recovery Date"] = rec_date_5y.date()
            rec_period_5y = (rec_date_5y - peak_date_5y).days
            etfs_data.loc[etf_name, "5 Year Recovery Period (days)"] = rec_period_5y

        # Calculate the two most significant maximum drawdown in the 5 year period
        drawdowns_5y = drawdowns_5y.sort_values()
        mdd_5y = drawdowns_5y.iloc[0] * 100

    # Calculate Calmar ratio
    calmar_ratio = (daily_returns.mean() * 252 - risk_free_rate) / abs(mdd / 100)
    etfs_data.loc[etf_name, "Max Period Calmar"] = calmar_ratio

    # Calculate 5 year Calmar ratio
    if len(df) >= 252 * 5:
        calmar_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / abs(mdd_5y / 100)
        etfs_data.loc[etf_name, "5 Year Calmar"] = calmar_ratio_5y

In [ ]:
etfs_data

In [ ]:
# Save the data as a CSV file
result_folder = "Result"
filename = os.path.join(result_folder, "etfs_comparison.csv")
etfs_data.to_csv(filename)